> English companion notebook for point-cloud sequence modeling. Code cells are kept aligned with the Chinese version.

# CNN + BiLSTM Point-Cloud Sequence Recognition

This notebook explains the model side of the mmLock-style pipeline. The signal-processing notebook produces point clouds; this notebook turns consecutive point-cloud frames into action labels.

A single frame is usually not enough to know whether someone is leaving. The model therefore reads a sequence of frames and learns the motion pattern over time.

Each point may contain fields such as:

| Field | Meaning |
| --- | --- |
| x, y, z | spatial position estimated from range and angle |
| velocity | Doppler-derived radial motion |
| intensity | reflection strength |
| frame/time | temporal position in the sequence |


## 1. Device Selection: CUDA / MPS / CPU

The notebook first chooses the compute device. CUDA is preferred on NVIDIA GPUs, MPS can be used on Apple Silicon, and CPU is the fallback.

This affects speed, not model meaning. The same tensor shapes and training logic should work across devices. If results differ strongly by device, check random seeds, deterministic settings, and numerical precision.


## 1.1 Storage Format: Per-Frame NPZ and Merged HDF5

Point-cloud data can be stored in two practical ways:

- **Per-frame NPZ**: easy to inspect and generate incrementally, but can be slow when there are many small files.
- **Merged HDF5**: faster for sequential loading and easier to package, but requires a preprocessing step.

For experimentation, NPZ is convenient. For repeated training, HDF5 is usually cleaner because one file can store many frames and labels with consistent indexing.


## 1.2 Optional: Merge Per-Frame NPZ Files into HDF5

The merge step converts many small point-cloud files into one structured dataset. This reduces file-system overhead and makes sequence sampling easier.

The key rule is to preserve frame order and label alignment. A sequence model depends on time order. If frames are shuffled or labels drift by one frame, training can look normal while the learned behavior is wrong.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import csv
import math
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = choose_device()
print("Using device:", DEVICE)


## 2. Training Configuration

Training configuration defines the input shape and learning process. Important values include:

- number of frames per action sample;
- maximum points per frame;
- number of features per point;
- batch size;
- learning rate;
- number of classes;
- train/validation split.

These are not arbitrary knobs. They encode assumptions about how long a leaving action lasts and how much point-cloud detail the model can process.


In [ ]:
@dataclass
class TrainConfig:
    # Parallel FFT output directory containing point_cloud_XXXXXX.npz files.
    point_cloud_dir: Path = Path("radar_fft_cube_progress_parallel/outputs/point_clouds")

    # Label file. The recommended format is described in the next section.
    labels_csv: Path = Path("radar_fft_cube_progress_parallel/outputs/labels.csv")

    # Number of frames in each action sample. The paper uses 30 point clouds for LSTM, so this default stays aligned.
    sequence_length: int = 30

    # Fixed number of sampled points per frame. PointNet commonly uses 2048; this CNN path also uses 2048 for consistency.
    points_per_frame: int = 2048

    # Features used for each point. Field names must exist in the structured points array inside point_cloud_XXXXXX.npz.
    point_features: tuple[str, ...] = (
        "range_m",
        "velocity_mps",
        "angle_deg",
        "power_db",
        "doppler_bin",
        "angle_bin",
        "range_bin",
    )

    # Model and training parameters.
    batch_size: int = 8
    num_epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    validation_ratio: float = 0.2
    random_seed: int = 42

    # Class order: output 0 means non_leaving, output 1 means leaving.
    class_names: tuple[str, ...] = ("non_leaving", "leaving")


cfg = TrainConfig()
cfg


## 3. Label File Format

The label file maps frame ranges or sequence IDs to action classes. For a leaving detector, labels may include stay, leave, approach, sit, or other motion categories depending on the dataset design.

Good labels should be:

- temporally aligned with radar frames;
- consistent across subjects and sessions;
- explicit about ambiguous transitions;
- reproducible from the raw recordings.

Weak labeling is one of the easiest ways to make a model look unstable.


In [ ]:
def load_label_rows(labels_csv: Path) -> list[dict]:
    if not labels_csv.exists():
        raise FileNotFoundError(
            f"Label file not found: {labels_csv}. "
            "Create labels.csv with columns: sample_id,start_frame,end_frame,label."
        )

    rows: list[dict] = []
    with labels_csv.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        required = {"sample_id", "start_frame", "end_frame", "label"}
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"labels.csv missing columns: {sorted(missing)}")

        for row in reader:
            rows.append(
                {
                    "sample_id": row["sample_id"],
                    "start_frame": int(row["start_frame"]),
                    "end_frame": int(row["end_frame"]),
                    "label": row["label"].strip(),
                }
            )
    return rows


## 4. Read `.npz` Point Clouds and Normalize to Fixed Size

Each frame can contain a different number of detected radar points. Neural networks need a fixed tensor shape, so the loader converts each frame to a fixed maximum number of points.

Common rules:

- If there are too many points, sample or select the most useful ones.
- If there are too few points, pad with zeros.
- Keep feature columns in a consistent order.
- Store a mask if the model needs to distinguish real points from padding.

This step turns sparse radar detections into trainable tensors.


In [ ]:
def point_cloud_path(point_cloud_dir: Path, frame_index: int) -> Path:
    return point_cloud_dir / f"point_cloud_{frame_index:06d}.npz"


def structured_points_to_array(points: np.ndarray, feature_names: Iterable[str]) -> np.ndarray:
    arrays = []
    for name in feature_names:
        if name not in points.dtype.names:
            raise KeyError(f"Feature {name!r} not found in point dtype: {points.dtype.names}")
        arrays.append(points[name].astype(np.float32))
    return np.stack(arrays, axis=1) if arrays else np.empty((len(points), 0), dtype=np.float32)


def sample_or_pad_points(frame_points: np.ndarray, target_points: int) -> np.ndarray:
    num_points, num_features = frame_points.shape
    if num_points == target_points:
        return frame_points

    if num_points == 0:
        return np.zeros((target_points, num_features), dtype=np.float32)

    if num_points > target_points:
        # power_db is the 4th feature; strong-reflection points are usually more stable.
        power_col = 3
        keep = np.argsort(frame_points[:, power_col])[-target_points:]
        return frame_points[keep].astype(np.float32)

    extra = target_points - num_points
    repeat_idx = np.random.choice(num_points, size=extra, replace=True)
    padded = np.concatenate([frame_points, frame_points[repeat_idx]], axis=0)
    return padded.astype(np.float32)


def load_one_frame(path: Path, cfg: TrainConfig) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(f"Point cloud file not found: {path}")

    with np.load(path, allow_pickle=False) as data:
        points = data["points"]

    frame_points = structured_points_to_array(points, cfg.point_features)
    frame_points = sample_or_pad_points(frame_points, cfg.points_per_frame)
    return frame_points


## 5. Feature Normalization

Radar point features have different scales. Position may be in meters, velocity in meters per second, and intensity in arbitrary magnitude units. Normalization prevents one feature from dominating only because its numeric range is larger.

Typical normalization options include z-score normalization, min-max scaling, and clipping extreme values. The same normalization statistics used during training must also be used during inference.

For security sensing, consistency is more important than elegance. A model trained with one feature scale should not be deployed with another.


## 4.1 Optional Code: Merge NPZ into HDF5

This optional code path packages point-cloud frames into HDF5. It is useful when training repeatedly or when the dataset contains many files.

The merged file should keep enough metadata to reconstruct the sample:

- frame index or timestamp;
- subject/session ID if available;
- action label;
- point tensor shape;
- feature order.

Without metadata, the model may still train, but later debugging becomes difficult.


In [ ]:
def merge_npz_point_clouds_to_h5(
    point_cloud_dir: Path,
    output_h5: Path,
    cfg: TrainConfig,
    compression_level: int = 1,
) -> Path:
    """Merge per-frame point_cloud_XXXXXX.npz files into one HDF5 file.

    HDF5 is useful when the storage medium has high random-seek cost. We use
    gzip level 1 because it is fast and still reduces file size.
    """
    try:
        import h5py
    except ImportError as exc:
        raise ImportError("h5py is required for HDF5 export. Install h5py first.") from exc

    npz_files = sorted(point_cloud_dir.glob("point_cloud_*.npz"))
    if not npz_files:
        raise FileNotFoundError(f"No point_cloud_*.npz files found in {point_cloud_dir}")

    output_h5.parent.mkdir(parents=True, exist_ok=True)
    num_frames = len(npz_files)
    num_features = len(cfg.point_features)

    with h5py.File(output_h5, "w") as h5:
        points_ds = h5.create_dataset(
            "points",
            shape=(num_frames, cfg.points_per_frame, num_features),
            dtype="float32",
            chunks=(1, cfg.points_per_frame, num_features),
            compression="gzip",
            compression_opts=compression_level,
            shuffle=True,
        )
        frame_index_ds = h5.create_dataset("frame_index", shape=(num_frames,), dtype="int64")
        num_points_ds = h5.create_dataset("num_points", shape=(num_frames,), dtype="int64")

        h5.attrs["point_features"] = ",".join(cfg.point_features)
        h5.attrs["points_per_frame"] = cfg.points_per_frame
        h5.attrs["compression"] = f"gzip:{compression_level}"

        for out_idx, npz_path in enumerate(npz_files):
            with np.load(npz_path, allow_pickle=False) as data:
                raw_points = data["points"]
                frame_index = int(data["frame_index"])
                num_points = int(data["num_points"])

            frame_points = structured_points_to_array(raw_points, cfg.point_features)
            frame_points = sample_or_pad_points(frame_points, cfg.points_per_frame)

            points_ds[out_idx] = frame_points
            frame_index_ds[out_idx] = frame_index
            num_points_ds[out_idx] = num_points

            if (out_idx + 1) % 1000 == 0:
                print(f"Merged {out_idx + 1}/{num_frames} frames")

    print("HDF5 saved:", output_h5)
    return output_h5


# Usage example:
# merge_npz_point_clouds_to_h5(
#     point_cloud_dir=cfg.point_cloud_dir,
#     output_h5=Path("radar_fft_cube_progress_parallel/outputs/radar_point_clouds.h5"),
#     cfg=cfg,
#     compression_level=1,
# )


## 4.2 Optional Code: Read Consecutive Frames from HDF5

Sequence models need consecutive frames. Reading from HDF5 should therefore return a time-ordered block, not a random collection of frames.

A sample shape is often:

`[sequence_length, max_points, feature_dim]`

The model then extracts spatial features per frame and temporal features across frames. This is the core difference between point-cloud classification and action recognition.


In [ ]:
class H5RadarPointSequenceDataset(Dataset):
    def __init__(self, h5_path: Path, rows: list[dict], cfg: TrainConfig, normalizer) -> None:
        self.h5_path = h5_path
        self.rows = rows
        self.cfg = cfg
        self.normalizer = normalizer
        self.class_to_index = {name: i for i, name in enumerate(cfg.class_names)}
        self._h5 = None

    def __len__(self) -> int:
        return len(self.rows)

    def _file(self):
        if self._h5 is None:
            import h5py

            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.rows[index]
        frame_indices = self._frame_indices(row["start_frame"], row["end_frame"])

        h5 = self._file()
        # Read frame by frame instead of using fancy indexing. This keeps h5py stable even when short sequences repeat frame indexes during padding.
        x = np.stack([h5["points"][frame_index] for frame_index in frame_indices], axis=0).astype(np.float32)
        x = self.normalizer(x).astype(np.float32)

        y = self.class_to_index[row["label"]]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

    def _frame_indices(self, start_frame: int, end_frame: int) -> list[int]:
        indices = list(range(start_frame, end_frame + 1))
        if len(indices) >= self.cfg.sequence_length:
            return indices[: self.cfg.sequence_length]
        while len(indices) < self.cfg.sequence_length:
            indices.append(indices[-1])
        return indices


In [ ]:
class PointFeatureNormalizer:
    def __init__(self, mean: np.ndarray, std: np.ndarray) -> None:
        self.mean = mean.astype(np.float32)
        self.std = np.maximum(std.astype(np.float32), 1e-6)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean[None, None, :]) / self.std[None, None, :]


def fit_normalizer(rows: list[dict], cfg: TrainConfig, max_frames: int = 300) -> PointFeatureNormalizer:
    # Sample only part of the frames to estimate mean and std, avoiding a slow full scan.
    all_frame_indices: list[int] = []
    for row in rows:
        all_frame_indices.extend(range(row["start_frame"], row["end_frame"] + 1))

    random.shuffle(all_frame_indices)
    all_frame_indices = all_frame_indices[:max_frames]

    chunks = []
    for frame_index in all_frame_indices:
        path = point_cloud_path(cfg.point_cloud_dir, frame_index)
        if path.exists():
            chunks.append(load_one_frame(path, cfg))

    if not chunks:
        num_features = len(cfg.point_features)
        return PointFeatureNormalizer(np.zeros(num_features), np.ones(num_features))

    values = np.concatenate(chunks, axis=0)
    return PointFeatureNormalizer(values.mean(axis=0), values.std(axis=0))


## 6. Dataset: Consecutive Frames as One Action Sample

The dataset class builds one training example from a window of consecutive frames. This window should be long enough to capture the leaving motion but short enough to avoid mixing unrelated actions.

Important dataset checks:

- Do all frames in a sample belong to the same labeled action?
- Are windows allowed to overlap?
- Does the train/validation split avoid leaking the same session into both sets?
- Are padded frames or empty frames handled consistently?

These checks often matter more than changing the model architecture.


In [ ]:
class RadarPointSequenceDataset(Dataset):
    def __init__(self, rows: list[dict], cfg: TrainConfig, normalizer: PointFeatureNormalizer) -> None:
        self.rows = rows
        self.cfg = cfg
        self.normalizer = normalizer
        self.class_to_index = {name: i for i, name in enumerate(cfg.class_names)}

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.rows[index]
        frame_indices = self._frame_indices(row["start_frame"], row["end_frame"])

        frames = []
        for frame_index in frame_indices:
            frame_path = point_cloud_path(self.cfg.point_cloud_dir, frame_index)
            frames.append(load_one_frame(frame_path, self.cfg))

        x = np.stack(frames, axis=0).astype(np.float32)
        x = self.normalizer(x).astype(np.float32)

        label = row["label"]
        if label not in self.class_to_index:
            raise ValueError(f"Unknown label {label!r}. Expected {self.cfg.class_names}.")
        y = self.class_to_index[label]

        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

    def _frame_indices(self, start_frame: int, end_frame: int) -> list[int]:
        indices = list(range(start_frame, end_frame + 1))
        if len(indices) >= self.cfg.sequence_length:
            return indices[: self.cfg.sequence_length]
        if not indices:
            raise ValueError("Empty frame range in labels.csv")
        while len(indices) < self.cfg.sequence_length:
            indices.append(indices[-1])
        return indices


## 7. Model Architecture: CNN + BiLSTM

The model has two conceptual parts:

1. A spatial encoder extracts features from each point-cloud frame.
2. A BiLSTM reads the feature sequence and models motion over time.

CNN here means the implementation uses convolution-like operations over a structured tensor representation of points. BiLSTM reads the sequence in both directions during training/inference, which can help when the full action window is available.

For online deployment, a causal LSTM may be more appropriate because future frames are not available yet. For offline evaluation, BiLSTM is a strong baseline.


In [ ]:
class FramePointCNN(nn.Module):
    def __init__(self, input_dim: int, embedding_dim: int = 256) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 128, kernel_size=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, embedding_dim, kernel_size=1),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch * time, points, features]
        x = x.transpose(1, 2)  # -> [batch * time, features, points]
        x = self.net(x)
        max_pool = torch.max(x, dim=-1).values
        avg_pool = torch.mean(x, dim=-1)
        return torch.cat([max_pool, avg_pool], dim=-1)


class CNNBiLSTMClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int = 2,
        cnn_embedding_dim: int = 256,
        lstm_hidden_dim: int = 128,
        lstm_layers: int = 2,
        dropout: float = 0.35,
    ) -> None:
        super().__init__()
        self.frame_encoder = FramePointCNN(input_dim, cnn_embedding_dim)
        frame_dim = cnn_embedding_dim * 2

        self.temporal_encoder = nn.LSTM(
            input_size=frame_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(lstm_hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, time, points, features]
        batch_size, time_steps, num_points, num_features = x.shape
        x = x.reshape(batch_size * time_steps, num_points, num_features)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(batch_size, time_steps, -1)

        temporal_features, _ = self.temporal_encoder(frame_features)

        # BiLSTM outputs bidirectional features for each time step. Use the last time step as the representation of the whole action segment.
        final_feature = temporal_features[:, -1, :]
        return self.classifier(final_feature)


## 8. Training and Validation Functions

The training loop updates model weights using labeled samples. The validation loop measures performance without gradient updates.

A useful validation report should include more than accuracy:

- confusion matrix across action classes;
- false positives that would lock the screen incorrectly;
- false negatives that would fail to lock after leaving;
- latency from action start to detection;
- performance by subject, angle, speed, and environment.

For mmLock, the cost of different mistakes is not symmetric. Missing a real leaving event is a security risk; false alarms hurt usability.


In [ ]:
def move_batch_to_device(batch: tuple[torch.Tensor, torch.Tensor], device: torch.device):
    x, y = batch
    return x.to(device), y.to(device)


def run_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    device: torch.device,
) -> dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in loader:
        x, y = move_batch_to_device(batch, device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        if is_train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += float(loss.detach().cpu()) * y.size(0)
        total_correct += int((preds == y).sum().detach().cpu())
        total_count += y.size(0)

    return {
        "loss": total_loss / max(total_count, 1),
        "accuracy": total_correct / max(total_count, 1),
    }


## 9. Build DataLoader and Model

The DataLoader batches sequence samples so the model receives tensors with consistent shape. The model is then moved to the selected device and optimized with the configured loss function.

At this stage, inspect one batch before training:

- tensor shape should match `[batch, time, points, features]` or the model's expected layout;
- labels should be integer class IDs;
- padding should not dominate every frame;
- feature values should be within the expected normalized range.

This catches many silent data bugs before training starts.


In [ ]:
def build_dataloaders(cfg: TrainConfig) -> tuple[DataLoader, DataLoader]:
    random.seed(cfg.random_seed)
    np.random.seed(cfg.random_seed)
    torch.manual_seed(cfg.random_seed)

    rows = load_label_rows(cfg.labels_csv)
    normalizer = fit_normalizer(rows, cfg)
    dataset = RadarPointSequenceDataset(rows, cfg, normalizer)

    if len(dataset) < 2:
        raise ValueError("At least two labeled segments are required for train/validation split.")

    val_size = max(1, int(len(dataset) * cfg.validation_ratio))
    train_size = max(1, len(dataset) - val_size)
    if train_size + val_size > len(dataset):
        train_size = len(dataset) - val_size

    generator = torch.Generator().manual_seed(cfg.random_seed)
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )
    return train_loader, val_loader


def build_model(cfg: TrainConfig) -> CNNBiLSTMClassifier:
    return CNNBiLSTMClassifier(
        input_dim=len(cfg.point_features),
        num_classes=len(cfg.class_names),
    ).to(DEVICE)


## 10. Start Training

Training should be treated as an experiment with diagnostics. Loss decreasing is necessary but not sufficient. The model may overfit one subject, one room, or one label artifact.

Recommended checks:

- compare training and validation curves;
- save the best validation checkpoint, not only the final epoch;
- inspect misclassified sequences;
- test with different sequence lengths;
- verify that validation data is separated by session or subject when possible.

The goal is not merely to fit the dataset. The goal is to detect leaving robustly under realistic variation.


In [ ]:
def train(cfg: TrainConfig) -> CNNBiLSTMClassifier:
    train_loader, val_loader = build_dataloaders(cfg)
    model = build_model(cfg)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    model_dir = Path("models")
    model_dir.mkdir(parents=True, exist_ok=True)
    best_path = model_dir / "cnn_blstm_best.pt"

    best_acc = -math.inf
    history = []

    for epoch in range(1, cfg.num_epochs + 1):
        train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        with torch.no_grad():
            val_metrics = run_one_epoch(model, val_loader, criterion, None, DEVICE)

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d} | "
            f"train loss {row['train_loss']:.4f} acc {row['train_accuracy']:.4f} | "
            f"val loss {row['val_loss']:.4f} acc {row['val_accuracy']:.4f}"
        )

        if row["val_accuracy"] > best_acc:
            best_acc = row["val_accuracy"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": cfg.__dict__,
                    "class_names": cfg.class_names,
                    "history": history,
                },
                best_path,
            )

    print("Best checkpoint:", best_path)
    return model


# Run training:
# model = train(cfg)


## 11. Inference on One Action Segment

Inference takes one sequence of point-cloud frames and returns class probabilities or a predicted label. For a user-leaving detector, deployment logic may also smooth predictions across windows to avoid flicker.

A practical decision rule might require several consecutive leave predictions before locking, or might use confidence thresholds. This trades off speed and false alarms.

The model output is therefore only one part of the final system decision.


In [ ]:
def predict_segment(
    model: CNNBiLSTMClassifier,
    cfg: TrainConfig,
    normalizer: PointFeatureNormalizer,
    start_frame: int,
    end_frame: int,
) -> dict[str, float | str]:
    row = {"sample_id": "prediction", "start_frame": start_frame, "end_frame": end_frame, "label": cfg.class_names[0]}
    dataset = RadarPointSequenceDataset([row], cfg, normalizer)
    x, _ = dataset[0]
    x = x.unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()

    pred_idx = int(np.argmax(probs))
    return {
        "prediction": cfg.class_names[pred_idx],
        "non_leaving_prob": float(probs[0]),
        "leaving_prob": float(probs[1]),
    }


## 12. Relationship to the FFT Results

This notebook depends on the radar-processing notebook. The model assumes that each point already represents a physically meaningful radar detection.

The full path is:

raw ADC samples -> Range FFT -> Doppler FFT -> Angle FFT -> detected points -> cleaned point-cloud frames -> sequence tensor -> CNN + BiLSTM -> action label.

If the FFT pipeline changes feature definitions or coordinate conventions, the model input changes too. The README and site should therefore link both the signal-processing notebook and the model-training notebook together.
